In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import math
import re
from datetime import datetime, timezone

# Not including API key here for security reasons
API_KEY = "API_KEY"

TOPIC_QUERIES = [
    "immigration policy",
    "healthcare policy",
    "housing policy",
    "labor policy",
    "climate policy"
]

VIDEOS_PER_TOPIC = 40
MAX_PER_SEARCH_CALL = 50

SEARCH_URL = "https://www.googleapis.com/youtube/v3/search"
VIDEOS_URL = "https://www.googleapis.com/youtube/v3/videos"

POLICY_TERMS = {
    "policy", "bill", "law", "senate", "congress", "regulation", "reform",
    "budget", "legislation", "border", "immigration", "healthcare", "housing",
    "labor", "climate", "tax", "vote", "rights", "court", "federal", "state"
}

SOURCE_TERMS = {
    "source", "sources", "report", "study", "research", "data", "article",
    "transcript", "read more", "citation", "citations", "link", "links"
}

def youtube_get(url, params):
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    if "error" in data:
        raise RuntimeError(data["error"])

    return data


def parse_iso8601_duration(duration_text):
    """
    Convert YouTube ISO 8601 duration like PT4M13S into total seconds.
    """
    pattern = r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?"
    match = re.fullmatch(pattern, duration_text)

    if not match:
        return None

    hours = int(match.group(1)) if match.group(1) else 0
    minutes = int(match.group(2)) if match.group(2) else 0
    seconds = int(match.group(3)) if match.group(3) else 0

    total_seconds = hours * 3600 + minutes * 60 + seconds
    return total_seconds


def clean_text(text):
    if text is None:
        return ""
    return str(text).strip()


def word_count(text):
    text = clean_text(text)
    if text == "":
        return 0
    words = re.findall(r"\b[\w'-]+\b", text.lower())
    return len(words)


def count_keyword_hits(text, keyword_set):
    text = clean_text(text).lower()
    words = re.findall(r"\b[\w'-]+\b", text)
    word_set = set(words)
    hits = word_set.intersection(keyword_set)
    return len(hits)


def has_link(text):
    text = clean_text(text).lower()
    return int(("http://" in text) or ("https://" in text) or ("www." in text))


def length_bucket(duration_seconds):
    """
    Buckets for your question:
    shorter vs longer content.
    """
    if duration_seconds is None:
        return "Unknown"
    if duration_seconds < 240:
        return "Short (<4 min)"
    elif duration_seconds < 600:
        return "Medium (4-10 min)"
    else:
        return "Long (10+ min)"


def safe_int(value):
    try:
        return int(value)
    except:
        return 0

def search_video_ids_for_query(query, target_n):
    """
    Use search.list to get video IDs for one topic.
    """
    all_rows = []
    next_page_token = None

    while len(all_rows) < target_n:
        params = {
            "part": "snippet",
            "q": query,
            "type": "video",
            "maxResults": min(MAX_PER_SEARCH_CALL, target_n - len(all_rows)),
            "order": "relevance",
            "safeSearch": "none",
            "key": API_KEY
        }

        if next_page_token is not None:
            params["pageToken"] = next_page_token

        data = youtube_get(SEARCH_URL, params)

        items = data.get("items", [])
        for item in items:
            video_id = item["id"].get("videoId")
            if video_id is None:
                continue

            row = {
                "topic_query": query,
                "video_id": video_id,
                "search_title": clean_text(item["snippet"].get("title", "")),
                "search_description": clean_text(item["snippet"].get("description", "")),
                "channel_title_from_search": clean_text(item["snippet"].get("channelTitle", "")),
                "published_at_from_search": item["snippet"].get("publishedAt", ""),
                "search_rank_within_query": len(all_rows) + 1
            }
            all_rows.append(row)

            if len(all_rows) >= target_n:
                break

        next_page_token = data.get("nextPageToken")
        if not next_page_token:
            break

    return pd.DataFrame(all_rows)


search_frames = []

for query in TOPIC_QUERIES:
    print(f"Searching: {query}")
    one_topic_df = search_video_ids_for_query(query, VIDEOS_PER_TOPIC)
    search_frames.append(one_topic_df)

search_df = pd.concat(search_frames, ignore_index=True)

search_df = search_df.drop_duplicates(subset=["video_id"]).reset_index(drop=True)

print("Unique videos found:", len(search_df))

def fetch_videos_metadata(video_ids):
    """
    Use videos.list to get snippet, statistics, and contentDetails.
    videos.list supports comma-separated IDs and is cheap in quota.
    """
    all_items = []

    chunk_size = 50
    for start in range(0, len(video_ids), chunk_size):
        chunk_ids = video_ids[start:start + chunk_size]

        params = {
            "part": "snippet,statistics,contentDetails",
            "id": ",".join(chunk_ids),
            "key": API_KEY
        }

        data = youtube_get(VIDEOS_URL, params)
        items = data.get("items", [])
        all_items.extend(items)

    rows = []
    for item in all_items:
        snippet = item.get("snippet", {})
        statistics = item.get("statistics", {})
        content_details = item.get("contentDetails", {})

        row = {
            "video_id": item.get("id", ""),
            "title": clean_text(snippet.get("title", "")),
            "description": clean_text(snippet.get("description", "")),
            "channel_title": clean_text(snippet.get("channelTitle", "")),
            "published_at": snippet.get("publishedAt", ""),
            "view_count": safe_int(statistics.get("viewCount", 0)),
            "like_count": safe_int(statistics.get("likeCount", 0)),
            "comment_count": safe_int(statistics.get("commentCount", 0)),
            "duration_iso8601": content_details.get("duration", "")
        }
        rows.append(row)

    return pd.DataFrame(rows)


metadata_df = fetch_videos_metadata(search_df["video_id"].tolist())

print("Metadata rows:", len(metadata_df))

df = search_df.merge(metadata_df, on="video_id", how="inner")

print("Merged rows:", len(df))

df["duration_seconds"] = df["duration_iso8601"].apply(parse_iso8601_duration)
df["duration_minutes"] = df["duration_seconds"] / 60
df["length_bucket"] = df["duration_seconds"].apply(length_bucket)

df["title_word_count"] = df["title"].apply(word_count)
df["description_word_count"] = df["description"].apply(word_count)

df["description_has_link"] = df["description"].apply(has_link)
df["policy_term_hits_in_description"] = df["description"].apply(lambda x: count_keyword_hits(x, POLICY_TERMS))
df["source_term_hits_in_description"] = df["description"].apply(lambda x: count_keyword_hits(x, SOURCE_TERMS))

df["informative_score"] = 0

df.loc[df["description_word_count"] >= 30, "informative_score"] += 1
df.loc[df["description_has_link"] == 1, "informative_score"] += 1
df.loc[df["policy_term_hits_in_description"] >= 2, "informative_score"] += 1
df.loc[df["source_term_hits_in_description"] >= 1, "informative_score"] += 1
df.loc[df["duration_seconds"] >= 600, "informative_score"] += 1

df["views_log10"] = df["view_count"].apply(lambda x: math.log10(x + 1))
df["likes_log10"] = df["like_count"].apply(lambda x: math.log10(x + 1))
df["comments_log10"] = df["comment_count"].apply(lambda x: math.log10(x + 1))

now = datetime.now(timezone.utc)

def days_since_published(published_text):
    try:
        dt = datetime.fromisoformat(published_text.replace("Z", "+00:00"))
        delta = now - dt
        return max(delta.days, 1)
    except:
        return None

df["days_since_published"] = df["published_at"].apply(days_since_published)
df["views_per_day"] = df["view_count"] / df["days_since_published"]
df["likes_per_day"] = df["like_count"] / df["days_since_published"]
df["comments_per_day"] = df["comment_count"] / df["days_since_published"]

df = df[df["length_bucket"] != "Unknown"].copy()
df = df[df["duration_seconds"].notna()].copy()

df.to_csv("youtube_policy_videos.csv", index=False)

print(df[[
    "topic_query", "title", "duration_minutes", "length_bucket",
    "description_word_count", "informative_score", "view_count"
]].head())

summary_by_length = (
    df.groupby("length_bucket")[[
        "description_word_count",
        "informative_score",
        "view_count",
        "views_per_day",
        "like_count",
        "comment_count"
    ]]
    .mean()
    .round(2)
    .sort_index()
)

summary_by_topic_and_length = (
    df.groupby(["topic_query", "length_bucket"])[[
        "description_word_count",
        "informative_score",
        "view_count",
        "views_per_day"
    ]]
    .mean()
    .round(2)
)

summary_by_length.to_csv("summary_by_length.csv")
summary_by_topic_and_length.to_csv("summary_by_topic_and_length.csv")

print("\nSummary by length bucket:")
print(summary_by_length)

print("\nSummary by topic and length bucket:")
print(summary_by_topic_and_length)

plot1 = (
    df.groupby("length_bucket")["informative_score"]
    .mean()
    .reindex(["Short (<4 min)", "Medium (4-10 min)", "Long (10+ min)"])
)

plt.figure(figsize=(8, 5))
plot1.plot(kind="bar")
plt.title("Average Informative Score by Video Length")
plt.xlabel("Video Length Group")
plt.ylabel("Average Informative Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("fig1_informative_score_by_length.png", dpi=300)
plt.show()

plot2 = (
    df.groupby("length_bucket")["views_per_day"]
    .mean()
    .reindex(["Short (<4 min)", "Medium (4-10 min)", "Long (10+ min)"])
)

plt.figure(figsize=(8, 5))
plot2.plot(kind="bar")
plt.title("Average Views Per Day by Video Length")
plt.xlabel("Video Length Group")
plt.ylabel("Average Views Per Day")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("fig2_views_per_day_by_length.png", dpi=300)
plt.show()

plt.figure(figsize=(8, 5))
plt.scatter(df["description_word_count"], df["views_log10"])
plt.title("Description Detail vs Video Views")
plt.xlabel("Description Word Count")
plt.ylabel("log10(View Count + 1)")
plt.tight_layout()
plt.savefig("fig3_description_words_vs_views.png", dpi=300)
plt.show()

plot4 = (
    df.groupby("topic_query")["informative_score"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(9, 5))
plot4.plot(kind="bar")
plt.title("Average Informative Score by Policy Topic")
plt.xlabel("Policy Topic")
plt.ylabel("Average Informative Score")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("fig4_informative_score_by_topic.png", dpi=300)
plt.show()

pivot_views = (
    df.pivot_table(
        index="topic_query",
        columns="length_bucket",
        values="views_per_day",
        aggfunc="mean"
    )
    .reindex(columns=["Short (<4 min)", "Medium (4-10 min)", "Long (10+ min)"])
)

plt.figure(figsize=(10, 6))
pivot_views.plot(kind="bar")
plt.title("Average Views Per Day by Topic and Video Length")
plt.xlabel("Policy Topic")
plt.ylabel("Average Views Per Day")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("fig5_views_per_day_by_topic_and_length.png", dpi=300)
plt.show()

print("\nPrototype checks")

length_compare = (
    df.groupby("length_bucket")[["informative_score", "views_per_day"]]
    .mean()
    .reindex(["Short (<4 min)", "Medium (4-10 min)", "Long (10+ min)"])
    .round(2)
)

print("\nMean informative score and views/day by length:")
print(length_compare)

topic_compare = (
    df.groupby("topic_query")[["informative_score", "views_per_day"]]
    .mean()
    .sort_values("views_per_day", ascending=False)
    .round(2)
)

print("\nMean informative score and views/day by topic:")
print(topic_compare)

Searching: immigration policy


HTTPError: 400 Client Error: Bad Request for url: https://www.googleapis.com/youtube/v3/search?part=snippet&q=immigration+policy&type=video&maxResults=40&order=relevance&safeSearch=none&key=API_KEY